# Cleaning Up Books

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter
from tqdm import tqdm
from langdetect import detect, DetectorFactory, LangDetectException

In [3]:
# defining paths
RAW = Path('../data/initial')

In [4]:
# loading everything in df
genres = ['children', 'comics_graphic', 'fantasy_paranormal', 'history_biography', 
          'mystery_thriller_crime', 'poetry', 'romance', 'young_adult']

paths = {g: RAW / f'goodreads_books_{g}.parquet' for g in genres}

frames = []
for genre, path in paths.items():
    part = pd.read_parquet(path)
    part['genre_file'] = genre
    print(f"{genre:<24} {len(part):>8,} rows  {len(part.columns):>3} cols")
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
del frames

children                  124,082 rows   30 cols
comics_graphic             89,411 rows   30 cols
fantasy_paranormal        258,585 rows   30 cols
history_biography         302,935 rows   30 cols
mystery_thriller_crime    219,235 rows   30 cols
poetry                     36,514 rows   30 cols
romance                   335,449 rows   30 cols
young_adult                93,398 rows   30 cols


In [5]:
df.shape

(1459609, 30)

In [6]:
df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series,genre_file
0,1599150603,7,[],US,,"[{'count': '4', 'name': 'history'}, {'count': ...",,false,4.13,B00DU10PUG,...,,2006,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,children
1,1934876569,6,[151854],US,,"[{'count': '25', 'name': 'fantasy'}, {'count':...",,false,4.22,,...,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,children
2,0590417010,193,[],US,eng,"[{'count': '64', 'name': 'picture-books'}, {'c...",,false,4.43,B017RORXNI,...,,1995,https://www.goodreads.com/book/show/89378.Dog_...,https://images.gr-assets.com/books/1360057676m...,89378,1331,86259,Dog Heaven,Dog Heaven,children
3,0915190575,4,[],US,,"[{'count': '1', 'name': 'kids-bookshelf'}, {'c...",,false,4.29,,...,,,https://www.goodreads.com/book/show/3209312-mo...,https://s.gr-assets.com/assets/nophoto/book/11...,3209312,11,3242879,"Moths and Mothers, Feathers and Fathers: A Sto...","Moths and Mothers, Feathers and Fathers: A Sto...",children
4,1416904999,4,[],US,,"[{'count': '4', 'name': 'board-books'}, {'coun...",,false,3.57,,...,,2005,https://www.goodreads.com/book/show/1698376.Wh...,https://s.gr-assets.com/assets/nophoto/book/11...,1698376,23,1695373,What Do You Do?,What Do You Do?,children


In [7]:
df = df.rename(columns={"genre_file": "genre"})

for c in ["ratings_count", "text_reviews_count", "average_rating",
          "num_pages", "publication_year", "publication_month", "publication_day"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df[["ratings_count", "average_rating", "num_pages", "publication_year"]].dtypes)

ratings_count         int64
average_rating      float64
num_pages           float64
publication_year    float64
dtype: object


In [8]:
df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series,genre
0,1599150603,7,[],US,,"[{'count': '4', 'name': 'history'}, {'count': ...",,false,4.13,B00DU10PUG,...,,2006.0,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,children
1,1934876569,6,[151854],US,,"[{'count': '25', 'name': 'fantasy'}, {'count':...",,false,4.22,,...,,2009.0,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,children
2,0590417010,193,[],US,eng,"[{'count': '64', 'name': 'picture-books'}, {'c...",,false,4.43,B017RORXNI,...,,1995.0,https://www.goodreads.com/book/show/89378.Dog_...,https://images.gr-assets.com/books/1360057676m...,89378,1331,86259,Dog Heaven,Dog Heaven,children
3,0915190575,4,[],US,,"[{'count': '1', 'name': 'kids-bookshelf'}, {'c...",,false,4.29,,...,,NaN,https://www.goodreads.com/book/show/3209312-mo...,https://s.gr-assets.com/assets/nophoto/book/11...,3209312,11,3242879,"Moths and Mothers, Feathers and Fathers: A Sto...","Moths and Mothers, Feathers and Fathers: A Sto...",children
4,1416904999,4,[],US,,"[{'count': '4', 'name': 'board-books'}, {'coun...",,false,3.57,,...,,2005.0,https://www.goodreads.com/book/show/1698376.Wh...,https://s.gr-assets.com/assets/nophoto/book/11...,1698376,23,1695373,What Do You Do?,What Do You Do?,children


## Tags

Getting tags and their counts.

In [9]:
def split_shelves(shelves):
    # if the book does not have any tags
    # return 2 empty lists
    if shelves is None or len(shelves) == 0:
        return [], []

    # if the book has shelves
    keep = [(str(s["name"]).strip().lower(), int(s["count"])) for s in shelves]
    return [t for t, _ in keep], [n for _, n in keep]

## Collapsing genre

Frankenstein can exist in both romance and fantasy.

In [ ]:
print(f"rows            {len(df):,}")
print(f"unique book_id  {df.book_id.nunique():,}")
print(f"duplicate rows  {df.book_id.duplicated().sum():,}")

rows            1,459,609
unique book_id  1,244,611
duplicate rows  214,998


In [ ]:
print(df.groupby("book_id").genre.nunique().value_counts().sort_index())

genre
1    1031796
2     210632
3       2183
Name: count, dtype: int64


In [ ]:
genres_per_book = df.groupby("book_id").genre.apply(list)

books = df.drop_duplicates("book_id").reset_index(drop=True)
books["genres"] = books.book_id.map(genres_per_book)

print(f"{len(books):,} unique books")
books[books.genres.apply(len) > 1][["title", "genres"]].head()

1,244,611 unique books


,title,genres
0,The Aeneid for Boys and Girls,"[children, history_biography]"
1,All's Fairy in Love and War (Avalon: Web of Ma...,"[children, fantasy_paranormal]"
8,"Katso eteesi, Lotta!","[children, young_adult]"
12,"Alice's adventures in Wonderland ; and, Throug...","[children, fantasy_paranormal]"
13,Growltiger's Last Stand and Other Poems,"[children, poetry]"


## Coverage

In [ ]:
def filled(s):
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        return s.fillna("").astype(str).str.strip().str.len().gt(0).mean()
    return s.notna().mean()

cols = ["description", "title", "isbn13", "isbn", "asin", "num_pages",
        "publication_year", "language_code", "average_rating", "url", "image_url"]

cov = pd.Series({c: filled(books[c]) for c in cols})
cov["tags"] = books.tags.apply(len).gt(0).mean()
cov.sort_values().map("{:.1%}".format)

asin                 21.1%
isbn                 59.9%
language_code        60.0%
isbn13               68.2%
num_pages            69.2%
publication_year     76.5%
description          89.0%
title               100.0%
average_rating      100.0%
url                 100.0%
image_url           100.0%
tags                100.0%
dtype: str

In [ ]:
books.groupby(pd.cut(books.ratings_count, [-1, 10, 100, 1000, 10**9]), observed=True) \
     .description.apply(lambda s: s.fillna("").str.len().gt(0).mean())

ratings_count
(-1, 10]              0.873624
(10, 100]             0.883262
(100, 1000]           0.905846
(1000, 1000000000]    0.949563
Name: description, dtype: float64

In [ ]:
popular = books[books.ratings_count > 1000]
no_desc = popular.description.fillna("").str.strip().str.len() == 0

print(f"books with >1000 ratings   {len(popular):,}")
print(f"  of those, no description {no_desc.sum():,}  ({no_desc.mean():.1%})")

books with >1000 ratings   69,810
  of those, no description 3,521  (5.0%)


In [ ]:
popular[no_desc][["title", "ratings_count", "tags"]] \
    .sort_values("ratings_count", ascending=False).head(20)

,title,ratings_count,tags
13897,"The Little House Collection (Little House, #1-9)",125070,"[classics, childrens, historical-fiction, fict..."
412813,Twenty Thousand Leagues Under the Sea,123184,"[fiction, classic, fantasy, literature, french..."
360890,"The Twilight Saga (Twilight, #1-4)",90307,"[fantasy, young-adult, romance, vampires, para..."
47256,Frindle,86161,"[realistic-fiction, childrens, fiction, childr..."
615160,My Horizontal Life: A Collection of One-Night ...,84293,"[humor, non-fiction, memoir, nonfiction, memoi..."
122125,The Little Engine That Could,82843,"[children-s, picture-books, childhood-favorite..."
152655,The Complete Persepolis,73851,"[graphic-novels, graphic-novel, comics, non-fi..."
201225,Anne Rice's The Vampire Lestat: A Graphic Novel,62825,"[horror, fantasy, graphic-novels, vampires, va..."
32839,"The Black Stallion (The Black Stallion, #1)",61170,"[classics, fiction, childrens, horses, young-a..."
1123463,Tara Road,60855,"[chick-lit, maeve-binchy, romance, oprah, fict..."


## Bundle detection

In [ ]:
def bundle_type(title):
    t = str(title).lower()
    if re.search(r"box(ed)?\s*set|boxset", t):        return "box set"
    if "collection" in t:                             return "collection"
    if "omnibus" in t:                                return "omnibus"
    if "complete" in t or "trilogy" in t:             return "complete/trilogy"
    if re.search(r"#\d+\s*[-–,]\s*\d+", t):           return "multi-volume range"
    if re.search(r"\bbooks?\s+\d+\s*[-–]\s*\d+", t):  return "multi-volume range"
    return "single title"

popular_nd = popular[no_desc].copy()
popular_nd["kind"] = popular_nd.title.apply(bundle_type)
popular_nd.kind.value_counts()

kind
single title          3348
complete/trilogy        69
multi-volume range      47
collection              28
box set                 27
omnibus                  2
Name: count, dtype: int64

## Editions

In [ ]:
print(f"book_id  {books.book_id.nunique():,}")
print(f"work_id  {books.work_id.nunique():,}")
print(f"work_id null/empty  {(books.work_id.fillna('') == '').sum():,}")

per_work = books.groupby("work_id").size()
print(f"\nmedian editions/work  {per_work.median():.0f}")
print(f"max                   {per_work.max():,}")
per_work.sort_values(ascending=False).head(10)

book_id  1,244,611
work_id  659,286
work_id null/empty  0

median editions/work  1
max                   520


work_id
3060926     520
1565818     435
55548884    399
2180358     398
2977639     368
3165724     321
1540236     314
4640799     244
1993810     235
3360164     221
dtype: int64

In [ ]:
print(f"works with 1 edition   {per_work.eq(1).mean():.1%}")
print(f"works with >5          {per_work.gt(5).sum():,}")
print(f"rows in multi-edition works  {per_work[per_work > 1].sum():,} of {len(books):,}")

works with 1 edition   65.4%
works with >5          24,182
rows in multi-edition works  813,442 of 1,244,611


In [ ]:
top_work = per_work.idxmax()
books[books.work_id == top_work][["title", "ratings_count", "publication_year", "language_code"]] \
    .sort_values("ratings_count", ascending=False).head(15)

,title,ratings_count,publication_year,language_code
1005320,Pride and Prejudice,2078406,2000.0,eng
1054917,Pride and Prejudice,28995,1983.0,eng
1041006,Pride and Prejudice,17631,2012.0,eng
1171688,Pride and Prejudice,11922,NaN,eng
1170591,Pride and Prejudice,10314,2003.0,eng
1087001,Pride and Prejudice,2962,2004.0,
1087002,Orgullo y prejuicio,2659,2006.0,spa
1135456,Pride and Prejudice,2455,2012.0,eng
1170678,Pride and Prejudice,2420,NaN,eng
1170590,Pride and Prejudice,2314,2004.0,eng


In [ ]:
books = books.copy()

# Empty strings are NOT skipped by .first() (NaN is), so description presence
# has to be an explicit sort key or a work inherits a blank description from
# its most-rated edition even when a smaller edition has one.
books["_has_desc"] = books.description.fillna("").str.strip().str.len().gt(0)

# work_id is blank on some rows. Without a synthetic key every one of them
# collapses into a single meaningless group.
_blank = books.work_id.fillna("").str.strip().eq("")
books["_work_key"] = books.work_id.where(~_blank, "nowork_" + books.book_id.astype(str))

# Prefer an English edition for the canonical title. Pride and Prejudice groups
# with Orgullo y prejuicio; without this the work can end up titled in Spanish
# purely because that edition happened to carry a description.
_lang = books.language_code.fillna("").str.strip().str.lower()
books["_is_en"] = _lang.str.startswith("en")

print(f"editions              {len(books):,}")
print(f"blank work_id         {_blank.sum():,}")
print(f"distinct work keys    {books._work_key.nunique():,}")

editions              1,244,611
blank work_id         0
distinct work keys    659,286


In [ ]:
books.to_parquet('../data/interim/books-before-canonical.parquet')

## Canonical Edition

In [ ]:
books = pd.read_parquet('../data/interim/books-before-canonical.parquet')

In [ ]:
works = (books.sort_values(
             ["_is_en", "_has_desc", "ratings_count"],
             ascending=[False, False, False],
             kind="mergesort")          # stable: ties keep file order, so reruns match
         .groupby("_work_key", as_index=False)
         .first())

print(f"works = {len(works):,}")

works = 659,286


In [ ]:
g = books.groupby("_work_key")

agg = g.agg(
    ratings_total=("ratings_count", "sum"),
    reviews_total=("text_reviews_count", "sum"),   # was inherited from one edition
    editions=("book_id", "size"),
    year_first=("publication_year", "min"),        # earliest = original publication
    pages_median=("num_pages", "median"),
    pages_min=("num_pages", "min"),
    pages_max=("num_pages", "max"),
)

# Weighted mean rating, vectorised. A 5.0 from one rater must not outweigh a
# 4.1 from 200,000, which a flat mean across editions would allow.
_w = books.ratings_count.fillna(0)
_prod = (books.average_rating.fillna(0) * _w).groupby(books._work_key).sum()
_den = _w.groupby(books._work_key).sum()
agg["rating_weighted"] = _prod / _den.replace(0, np.nan)

works = works.merge(agg, on="_work_key", how="left")

print(f"pages disagree  {(works.num_pages.sub(works.pages_median).abs() > 50).mean():.1%} of works by >50pp")

pages disagree  2.2% of works by >50pp


## Merging tags

In [ ]:
def _merge_tags(tags_series, counts_series):
    c = Counter()
    for tags, counts in zip(tags_series, counts_series):
        for t, n in zip(tags, counts):
            c[t] += int(n)
    return c.most_common()

_merged = {
    key: _merge_tags(grp.tags, grp.tag_counts)
    for key, grp in tqdm(books.groupby("_work_key")[["tags", "tag_counts"]])
}

works["tags"]       = works._work_key.map(lambda k: [t for t, _ in _merged[k]])
works["tag_counts"] = works._work_key.map(lambda k: [n for _, n in _merged[k]])

100%|██████████| 659286/659286 [01:09<00:00, 9422.79it/s] 


In [ ]:
print(works.tags.apply(len).describe())
print(works[["title", "editions"]].assign(top=works.tags.str[:5]).head().to_string())

count    659286.000000
mean         51.831597
std          32.231555
min           1.000000
25%          21.000000
50%          48.000000
75%          87.000000
max         161.000000
Name: tags, dtype: float64
                                                                                                                          title  editions                                                                   top
0                                                                                                        Treasury of Love Poems         1  [poetry, poland, my-reading-challenge, priority-literature, classic]
1                                                                                                        Siege: Malta 1940-1943         2                            [history, non-fiction, ww2, war, military]
2                                                                                           Julius Caesar: The Pursuit of Power         1         [biography, history

# Genre and IDs

In [ ]:
# A work can appear in several genre files, and different editions may land in
# different ones. Union rather than inheriting the canonical edition's.
_genres = books.groupby("_work_key").genre.apply(lambda s: sorted(set(s.dropna())))
works["genres"] = works._work_key.map(_genres)

print(works.genres.apply(len).value_counts().sort_index())

genres
1    659243
2        43
Name: count, dtype: int64


In [ ]:
work_book_ids = (books.groupby("_work_key").book_id
                      .apply(lambda x: sorted(set(x.astype(str)))))

works["book_id_all"] = works._work_key.map(work_book_ids)
works["book_id_all"] = works.book_id_all.apply(lambda v: v if isinstance(v, list) else [])

print(works.book_id_all.apply(len).sum(), "editions accounted for")

1244611 editions accounted for


## ID check

In [ ]:
book_to_work = books[["book_id", "_work_key"]].drop_duplicates()

# Every edition's identifiers, for a later external merge. Coverage per work is
# much better than per edition: a work with 40 editions only needs one to carry
# an ASIN. Amazon's edition is rarely the one Goodreads rated most.
def _key_list(col):
    s = books[books[col].fillna("").astype(str).str.strip().ne("")]
    return s.groupby("_work_key")[col].apply(lambda x: sorted(set(x.astype(str))))

work_keys = pd.DataFrame({
    "isbn13":      _key_list("isbn13"),
    "isbn":        _key_list("isbn"),
    "asin":        _key_list("asin"),
    "kindle_asin": _key_list("kindle_asin"),
}).reindex(works._work_key).reset_index()

cols = ["isbn13", "isbn", "asin", "kindle_asin"]
work_keys[cols] = work_keys[cols].map(lambda v: v if isinstance(v, list) else [])

for c in cols:
    per_edition = books[c].fillna("").astype(str).str.strip().ne("").mean()
    per_work = work_keys[c].apply(len).gt(0).mean()
    print(f"{c:12} {per_edition:6.1%} of editions -> {per_work:6.1%} of works")

isbn13        68.2% of editions ->  75.6% of works
isbn          59.9% of editions ->  68.6% of works
asin          21.1% of editions ->  34.5% of works
kindle_asin   47.4% of editions ->  54.3% of works


In [ ]:
assert works._work_key.is_unique, "work keys are not unique"
assert len(works) == books._work_key.nunique(), "lost or gained works"

_r_in, _r_out = books.ratings_count.sum(), works.ratings_total.sum()
_t_in = sum(sum(c) for c in books.tag_counts)
_t_out = sum(sum(c) for c in works.tag_counts)
assert np.isclose(_r_in, _r_out), f"ratings mismatch {_r_in:,} vs {_r_out:,}"
assert _t_in == _t_out, f"tag volume mismatch {_t_in:,} vs {_t_out:,}"

_naive = (books.sort_values("ratings_count", ascending=False)
               .groupby("_work_key", as_index=False).first())

print(f"\n{len(books):,} editions -> {len(works):,} works")
print(f"ratings preserved     {_r_in:,.0f}")
print(f"tag volume preserved  {_t_in:,}")
print(f"description coverage  "
      f"{works.description.fillna('').str.strip().str.len().gt(0).mean():.1%} "
      f"(naive sort would give "
      f"{_naive.description.fillna('').str.strip().str.len().gt(0).mean():.1%})")
print(f"non-english titles    "
      f"{(~works.language_code.fillna('').str.lower().str.startswith('en')).mean():.1%}")
print(f"num_pages null        {works.num_pages.isna().mean():.1%}")

works = works.drop(columns=["_has_desc", "_is_en"])


1,244,611 editions -> 659,286 works
ratings preserved     707,147,891
tag volume preserved  1,856,797,343
description coverage  86.3% (naive sort would give 85.0%)
non-english titles    45.4%
num_pages null        23.9%


## Language

In [ ]:
lang = works.language_code.fillna("").str.strip().str.lower().replace("", "(missing)")
print(lang.value_counts().head(20).to_string())

language_code
eng          289161
(missing)    247794
en-us         42050
en-gb         24732
ind            5697
fre            4903
ger            4442
spa            4330
ita            4201
en-ca          4164
ara            3757
jpn            3074
nl             2389
fin            2165
por            2063
swe            1661
per            1263
pol             974
gre             882
tur             798


In [ ]:
def lang_group(c):
    if c == "(missing)":   return "missing"
    if c.startswith("en"): return "english"
    return "other"

groups = lang.apply(lang_group)
print(groups.value_counts().to_string())
print()
print((groups.value_counts(normalize=True) * 100).round(1).to_string())

language_code
english    360234
missing    247794
other       51258

language_code
english    54.6
missing    37.6
other       7.8


In [ ]:
# is missing concentrated in obscure books? (scraping gap vs real)
works.groupby(pd.cut(works.ratings_total, [-1, 10, 100, 1000, 10**9]), observed=True) \
     .language_code.apply(lambda s: s.fillna("").str.strip().eq("").mean())

ratings_total
(-1, 10]              0.530162
(10, 100]             0.456490
(100, 1000]           0.289124
(1000, 1000000000]    0.083845
Name: language_code, dtype: float64

In [ ]:
# eyeball the most popular ones with no language code
works[works.language_code.fillna("").str.strip().eq("")] \
    [["title", "ratings_total"]].sort_values("ratings_total", ascending=False).head(25)

,title,ratings_total
237817,"Dr. Seuss's Green Eggs and Ham: For Soprano, B...",91678
349488,Janet Evanovich Three and Four Two-Book Set (S...,64776
594865,The Cat in the Hat and Other Dr. Seuss Favorites,46786
165161,There Was an Old Lady Who Swallowed a Fly,42800
337224,The Napping House,38909
417101,The Road to the Dark Tower: Exploring Stephen ...,32867
627361,Llama Llama Red Pajama,31929
502183,Harry Potter and the Chamber of Secrets: Sheet...,29972
96906,The Ripper (The Vampire Diaries: Stefan's Diar...,28432
499897,Sookie Stackhouse 7-copy Boxed Set (Sookie Sta...,25700


In [ ]:
works = works.merge(
    work_keys.rename(columns={"isbn13":      "isbn13_all",
                              "isbn":        "isbn_all",
                              "asin":        "asin_all",
                              "kindle_asin": "kindle_asin_all"}),
    on="_work_key", how="left",
)

# left join yields NaN for any work missing from work_keys; normalize to []
for c in ["isbn13_all", "isbn_all", "asin_all", "kindle_asin_all"]:
    works[c] = works[c].apply(lambda v: v if isinstance(v, list) else [])

assert len(works) == works._work_key.nunique(), "merge duplicated rows"

print(works[["title", "editions", "isbn13", "isbn13_all"]].head(5).to_string())

                                                                                                                          title  editions         isbn13                                     isbn13_all
0                                                                                                        Treasury of Love Poems         1  9780781806527                                [9780781806527]
1                                                                                                        Siege: Malta 1940-1943         2  9780850529302                                [9780850529302]
2                                                                                           Julius Caesar: The Pursuit of Power         1                                                            []
3                                                                                                      Jean-Paul Sartre: A Life         3  9781565849747  [9781565849747, 9788435026802, 9789100554965]


In [ ]:
DetectorFactory.seed = 0   # langdetect is nondeterministic without this

def detect_lang(text, min_chars=25):
    text = (text or "").strip()
    if len(text) < min_chars:
        return None
    try:
        return detect(text)
    except LangDetectException:
        return None

In [ ]:
blank = works.language_code.fillna("").str.strip().eq("")

known = works[~blank].sample(2000, random_state=0).copy()
known["detected"] = known.description.apply(detect_lang)
ok = known.dropna(subset=["detected"])
agree = (ok.detected.str[:2] == ok.language_code.str[:2].str.lower()).mean()
print(f"agreement: {agree:.1%}  (on {len(ok):,} of 2,000)")

agreement: 90.1%  (on 1,816 of 2,000)


In [ ]:
tqdm.pandas()

detected = works.loc[blank, "description"].progress_apply(detect_lang)
print(detected.value_counts(dropna=False).head(10).to_string())

100%|██████████| 247794/247794 [07:53<00:00, 522.88it/s]

description
en     191095
NaN     54265
es        484
fr        393
de        254
cy        235
id        211
it        125
nl         98
pt         95


In [ ]:
works["lang"] = works.language_code.fillna("").str.strip().str[:2].str.lower().replace("", np.nan)
works.loc[blank, "lang"] = detected.str[:2]
works["lang_source"] = np.where(blank, "detected", "goodreads")

print(works.lang.value_counts(dropna=False).head(10).to_string())
print(f"\nstill unknown: {works.lang.isna().mean():.1%}")

lang
en     551329
NaN     54265
in       5698
fr       5301
ge       4442
sp       4330
it       4326
ar       3759
jp       3074
po       3037

still unknown: 8.2%


In [ ]:
before = len(works)

keep = works.lang.eq("en")
english = works[keep].reset_index(drop=True)

print(f"{before:,} -> {len(english):,} works  ({keep.mean():.1%} kept)")
print()
print("dropped:")
print(f"  non-english  {works.lang.notna().sum() - keep.sum():,}")
print(f"  unknown      {works.lang.isna().sum():,}")

659,286 -> 551,329 works  (83.6% kept)

dropped:
  non-english  53,692
  unknown      54,265


In [ ]:
print(english.shape)
print(english.columns.tolist())
english.head()

(551329, 49)
['_work_key', 'isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series', 'genre', 'tags', 'tag_counts', 'genres', 'ratings_total', 'reviews_total', 'editions', 'year_first', 'pages_median', 'pages_min', 'pages_max', 'rating_weighted', 'book_id_all', 'isbn13_all', 'isbn_all', 'asin_all', 'kindle_asin_all', 'lang', 'lang_source']


,_work_key,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series,genre,tags,tag_counts,genres,ratings_total,reviews_total,editions,year_first,pages_median,pages_min,pages_max,rating_weighted,book_id_all,isbn13_all,isbn_all,asin_all,kindle_asin_all,lang,lang_source
0,1000008,0781806526,7,[],US,eng,"[{'count': '9', 'name': 'poetry'}, {'count': '...",,false,4.03,B00EK111RG,[],This beautiful bilingual gift edition contains...,Hardcover,https://www.goodreads.com/book/show/1013884.Tr...,"[{'author_id': '209272', 'role': ''}]",Hippocrene Books,137.0,1.0,9780781806527,5.0,,1998.0,https://www.goodreads.com/book/show/1013884.Tr...,https://s.gr-assets.com/assets/nophoto/book/11...,1013884,34,1000008,Treasury of Love Poems,Treasury of Love Poems,poetry,"[poetry, poland, my-reading-challenge, priorit...","[9, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",[poetry],34,7,1,1998.0,137.0,137.0,137.0,4.03,[1013884],[9780781806527],[0781806526],[],[B00EK111RG],en,goodreads
1,1000014,0850529301,5,[],US,en-US,"[{'count': '10', 'name': 'history'}, {'count':...",,false,3.87,,[],"Situated halfway between Europe and Africa, Ma...",Paperback,https://www.goodreads.com/book/show/1013890.Siege,"[{'author_id': '7359866', 'role': ''}]",Pen & Sword,248.0,28.0,9780850529302,4.0,,2003.0,https://www.goodreads.com/book/show/1013890.Siege,https://s.gr-assets.com/assets/nophoto/book/11...,1013890,37,1000014,Siege: Malta 1940-1943,Siege: Malta 1940-1943,history_biography,"[history, non-fiction, ww2, war, military, nav...","[20, 6, 6, 4, 4, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2,...",[history_biography],57,10,2,2003.0,271.5,248.0,295.0,3.87,"[1013890, 18160921]",[9780850529302],[0850529301],[B00DBXRAMY],[],en,goodreads
2,100003,1565849744,6,[],US,,"[{'count': '20', 'name': 'biography'}, {'count...",,false,3.82,,"[7905119, 1813909, 17271, 16097604, 17771043, ...",One of the major accomplishments of Cohen-Sola...,Paperback,https://www.goodreads.com/book/show/103716.Jea...,"[{'author_id': '59948', 'role': ''}, {'author_...","New Press, The",602.0,16.0,9781565849747,5.0,,2005.0,https://www.goodreads.com/book/show/103716.Jea...,https://images.gr-assets.com/books/1328752665m...,103716,74,100003,Jean-Paul Sartre: A Life,Jean-Paul Sartre: A Life,history_biography,"[biography, philosophy, non-fiction, jean-paul...","[60, 42, 27, 9, 9, 9, 6, 6, 6, 6, 6, 3, 3, 3, ...",[history_biography],79,8,3,1992.0,637.0,602.0,765.0,3.82,"[103716, 22724695, 688564]","[9781565849747, 9788435026802, 9789100554965]","[1565849744, 8435026809, 9100554960]",[],[],en,detected
3,100004,1555532519,2,[],US,,"[{'count': '3', 'name': 'non-fiction'}, {'coun...",,false,3.10,,[],"In this intimate memoir, Bianca Lamblin tells ...",Hardcover,https://www.goodreads.com/book/show/103717.A_D...,"[{'author_id': '59950', 'role': ''}, {'author_...",Northeastern University Press,224.0,1.0,9781555532512,3.0,,1996.0,https://www.goodreads.com/book/show/103717.A_D...,https://s.gr-assets.com/assets/nophoto/book/11...,103717,17,100004,"A Disgraceful Affair: Simone de Beauvoir, Jean...","A Disgraceful Affair: Simone de Beauvoir, Jean...",history_biography,"[non-fiction, biographies, history, defaut, au...","[3, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",[history_biography],17,2,1,1996.0,224.0,224.0,224.0,3.10,[103717],[9781555532512],[1555532519],[],[],en,detected
4,1000065,1588464792,3,[703515],US,eng,"[{'count': '8', 'name': 'rpg'}, {'count': '4',...",,false,3.81,,[],,Hardcover,https://www.goodreads.com/book/show/1013941.Wo...,"[{'author_id': '14142152', 'role': ''}, {'auth...",White Wolf Publishing,422.0,1.0,9781588464798,12.0,,2005.0,https://www.goodreads.com/book/show/1013941.Wo...,https://s.gr-assets.com/assets/nophoto/book/11...,1013941,53,1000065,Wo

In [ ]:
print(english.lang.value_counts(dropna=False))   # should be only 'en'
print(works.lang.isna().sum(), "unknown still in works")

lang
en    551329
Name: count, dtype: int64
54265 unknown still in works


In [ ]:
INTERIM = Path("../data/interim"); INTERIM.mkdir(parents=True, exist_ok=True)

english.to_parquet(INTERIM / "english_wip.parquet", index=False)
works.to_parquet(INTERIM / "works_wip.parquet", index=False)
book_to_work.to_parquet(INTERIM / "book_to_work.parquet", index=False)
books.to_parquet(INTERIM / "editions.parquet", index=False)

print(f"{len(english):,} english / {len(works):,} works / {len(book_to_work):,} editions")

551,329 english / 659,286 works / 1,244,611 editions


## start here

In [ ]:
english      = pd.read_parquet(INTERIM / "english_wip.parquet")
works        = pd.read_parquet(INTERIM / "works_wip.parquet")
book_to_work = pd.read_parquet(INTERIM / "book_to_work.parquet")

english.isna().sum()

In [ ]:
english = english.drop(columns=["work_id"]).rename(columns={"_work_key": "work_id"})

assert english.work_id.is_unique, "work_id not unique after rename"
print(english.columns.tolist())

['work_id', 'isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'title', 'title_without_series', 'genre', 'tags', 'tag_counts', 'genres', 'ratings_total', 'reviews_total', 'editions', 'year_first', 'pages_median', 'pages_min', 'pages_max', 'rating_weighted', 'book_id_all', 'isbn13_all', 'isbn_all', 'asin_all', 'kindle_asin_all', 'lang', 'lang_source']


In [ ]:
a = english.title.fillna("").str.strip()
b = english.title_without_series.fillna("").str.strip()

print(f"exactly the same   {(a == b).mean():.2%}   ({(a == b).sum():,} rows)")
print(f"differ             {(a != b).mean():.2%}   ({(a != b).sum():,} rows)")
print()
print(f"title null/blank   {a.eq('').sum():,}")
print(f"tws   null/blank   {b.eq('').sum():,}")
print()
print(english[a != b][["title", "title_without_series"]].head(10).to_string())

exactly the same   100.00%   (551,329 rows)
differ             0.00%   (0 rows)

title null/blank   2
tws   null/blank   2

Empty DataFrame
Columns: [title, title_without_series]
Index: []


In [ ]:
for q in ["Harry Potter", "Hunger Games", "Game of Thrones"]:
    sub = english[english.title.str.contains(q, case=False, na=False)]
    print(f"=== {q}: {len(sub)} rows")
    print(sub[["title", "title_without_series", "series", "ratings_total"]]
          .sort_values("ratings_total", ascending=False)
          .head(10).to_string())
    print()

=== Harry Potter: 87 rows
                                                                           title                                                      title_without_series     series  ratings_total
391848                  Harry Potter and the Sorcerer's Stone (Harry Potter, #1)                  Harry Potter and the Sorcerer's Stone (Harry Potter, #1)   [167817]        4970387
192773               Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)               Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)   [162083]        2016007
494259                Harry Potter and the Chamber of Secrets (Harry Potter, #2)                Harry Potter and the Chamber of Secrets (Harry Potter, #2)   [169291]        1950555
267990                    Harry Potter and the Goblet of Fire (Harry Potter, #4)                    Harry Potter and the Goblet of Fire (Harry Potter, #4)   [164829]        1909895
264758                   Harry Potter and the Deathly Hallows (Harry 

In [ ]:
before = len(english)

blank_title = english.title.fillna("").str.strip().eq("")
english = english[~blank_title].drop(columns=["title_without_series"]).reset_index(drop=True)

print(f"{before:,} -> {len(english):,}  (dropped {blank_title.sum()} blank-title rows)")
print(english.columns.tolist())

551,329 -> 551,327  (dropped 2 blank-title rows)
['work_id', 'isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'title', 'genre', 'tags', 'tag_counts', 'genres', 'ratings_total', 'reviews_total', 'editions', 'year_first', 'pages_median', 'pages_min', 'pages_max', 'rating_weighted', 'book_id_all', 'isbn13_all', 'isbn_all', 'asin_all', 'kindle_asin_all', 'lang', 'lang_source']


In [30]:
out = INTERIM / "english_works.parquet"

english.to_parquet(out, index=False)
print(f"{len(english):,} rows -> {out}")
print(f"{out.stat().st_size / 1e6:.1f} MB")

# verify it reads back intact before trusting it
check = pd.read_parquet(out)
assert len(check) == len(english), "row count changed"
assert check.work_id.is_unique, "work_id not unique after round-trip"
assert list(check.columns) == list(english.columns), "columns changed"
print("verified")

551,327 rows -> ../data/interim/english_works.parquet
980.9 MB
verified


In [31]:
book_to_work.to_parquet(INTERIM / "book_to_work.parquet", index=False)
print(f"{len(book_to_work):,} editions mapped")

1,244,611 editions mapped
